# 01: Data load: raw delivery to tidy store

Turns the 12 delivered Parquet files into `se_interval`: one row per site per 5-minute interval, in the CICCADA convention, partitioned and sorted for querying.

Every conversion happens here, once:

| | |
|---|---|
| **Time** | naive local civil string (with DST) → `ts_utc` → `ts_aest` (fixed UTC+10) |
| **Sign** | reactive power flipped from the load convention to the generator convention |
| **Units** | W → kW, var → kvar (instantaneous, *not* ×12) |
| **Shape** | per-phase columns folded to site level, phase detail kept separately |
| **Layout** | re-partitioned by AEST month, sorted by `(site_alias, ts_utc)` |

## Setup

The build should take roughly 30~45 s per month. 

Peak memory is about the DuckDB limit plus 200 MB, flat across the run, because each month gets its own short-lived connection.

If you are sharing the compute server, set `CICCADA_SE_DUCKDB_MEMORY` and
`CICCADA_SE_DUCKDB_THREADS` before importing. Left unset, DuckDB sizes itself from the
machine. If your raw data sits in OneDrive, consider `CICCADA_SE_STORE_DIR` pointing at
a local scratch disk — the store is ~1.5 GB and rebuilding it would otherwise push the
whole thing through cloud sync each time.

In [ ]:
# Reload edited .py modules without restarting the kernel.
# The library carries all the logic, so it gets edited constantly while notebooks
# stay thin -- without this, every module change means a kernel restart and a
# re-run of the expensive cells.
#
# CAVEAT: autoreload rebinds FUNCTIONS, not objects already constructed at import
# time. If you change a dataclass in se_params (SEAnalysisConfig, SEVoltVarParams)
# or a constant in se_config, restart the kernel -- `CONFIG` and `PARAMS` are
# module-level instances and will still be the old ones.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store
from solar_edge.lib import se_ingest as ing

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect(verbose=True)
print("Store directory:", C.STORE_DIR)

## 1. Conventions being applied

In [ ]:
display(C.describe_conventions())

## 2. Daylight-saving hazards in the raw delivery

**NSW and South Australia do observe daylight saving. Queensland does not.** 

The two windows below are hazardous *for NSW and SA*.

**October — clocks go forward.** At 02:00 on 2025-10-05, NSW and SA clocks jump straight
to 03:00. The wall-clock hour 02:00–02:59 **never happens** in those states on that date.
So a NSW or SA row carrying that timestamp could only come from a logger whose clock is
not daylight-saving aware. Left alone, ICU shifts it forward onto 03:00–03:59, where it
collides with a genuine reading — and deduplication would then throw away real data.

**April — clocks go back.** At 03:00 on 2025-04-06, NSW and SA clocks fall back to 02:00.
The wall-clock hour 02:00–02:59 therefore happens **twice**: once on daylight time, then
again an hour later on standard time. Two genuinely different moments wearing the same
label. For Sydney:

| Wall clock | Offset | Actual instant (UTC) |
|---|---|---|
| 02:30 (first pass, AEDT) | UTC+11 | 2025-04-05 15:30 |
| 02:30 (second pass, AEST) | UTC+10 | 2025-04-05 16:30 |

A bare string `2025-04-06 02:30:00` cannot tell you which. A site logging every 5 minutes
through the whole overlap emits 24 rows carrying only 12 distinct timestamps.

**Neither hazard materialises in this delivery.** Zero NSW/SA rows fall in the October
gap, so the site clocks are daylight-saving aware. And every April-overlap row has a
distinct key, so no site logged through both passes. Queensland rows show up in both
windows only because 02:00–03:00 is an entirely ordinary hour there.

`build_month` asserts this per month rather than trusting it — run this against any new
delivery before rebuilding.

In [ ]:
display(ing.dst_hazards(con))

## 3. Build the store

One month at a time. `build_store()` refuses to overwrite an existing store unless told to.

Passing `months=[...]` instead appends just those months, which makes an interrupted build resumable rather than a restart.

Each month writes files named for its source, so a month legitimately spills into the
neighbouring AEST partition without clobbering anything. That spillover is expected: a
South Australian midnight in local time is the previous day in the AEST analysis frame.

In [ ]:
# Set overwrite=True to rebuild from scratch. To resume after an interruption:
#     ing.build_store(con, months=["2025-07", "2025-08"])
stats = ing.build_store(con, overwrite=True)
display(stats)

print(f"Total rows written: {stats.store_rows.sum():,}")
print(f"Duplicate rows removed: {stats.duplicates_removed.sum():,}")
print(f"UTC collisions (must be 0): {stats.utc_collisions.sum()}")
print(f"Wall time: {stats.seconds.sum() / 60:.1f} min")

## 4. Reconciliation

Five claims, each checked against the **deduplicated** raw basis — comparing against raw
totals that still contain duplicates would always show a spurious mismatch.

1. **Rows accounted for** — an identity, not a tolerance: store rows equal raw distinct
   `(site, timestamp)` keys.
2. **Conflicting duplicates** — reported, not asserted (see section 6).
3. **Sites conserved** — 1,602 in both.
4. **Active energy conserved** — exercises the phase folding and the W→kW conversion.
5. **Reactive energy sign-flipped** — the store total must equal −1 × the raw total. This
   is the sign correction stated as a testable claim rather than a comment.

In [ ]:
recon = ing.reconcile(con)
display(recon)

assert recon["pass"].all(), "Reconciliation failed — do not proceed to D4."
print("Reconciliation PASSED.")

## 5. Did the timezone resolution work?

The fleet-wide diurnal profile in the AEST analysis frame is the acceptance test. It
should peak at hour 12 and fall to near zero overnight, with no double-humping or
smearing — which is what you would see if the three states had been pooled in
mismatched frames.

In [ ]:
profile = se_store.q(con, """
    SELECT hour(ts_aest) AS hour_aest,
           count(*)                  AS n_rows,
           round(avg(P_kW), 3)       AS mean_P_kW
    FROM se_interval
    GROUP BY 1 ORDER BY 1
""")

ax = profile.plot(x="hour_aest", y="mean_P_kW", kind="bar", figsize=(11, 3.4),
                  legend=False, color="#e8702a", width=0.85)
ax.set_xlabel("Hour of day (AEST, fixed UTC+10)")
ax.set_ylabel("Mean P (kW)")
ax.set_title("Fleet diurnal profile — peak should sit at hour 12")
peak = int(profile.loc[profile.mean_P_kW.idxmax(), "hour_aest"])
print(f"Peak hour: {peak}  (expected 12)")

### Per-state check

The stronger test. Each state was converted through its own IANA zone, so if that worked
the power-weighted centroids should land on each city's true solar noon expressed in
AEST — Brisbane earliest, then Sydney, then Adelaide about 50 minutes later.

The seasonal comparison is what proves DST was handled: Queensland must barely move
between June and January, while NSW and SA must not shift by a whole hour. If daylight
saving had been ignored, they would.

In [ ]:
centroids = se_store.q(con, """
    SELECT state,
           CASE WHEN month(ts_aest) = 6 THEN 'June' ELSE 'January' END AS season,
           round(sum((hour(ts_aest) * 60 + minute(ts_aest)) * greatest(P_kW, 0))
                 / nullif(sum(greatest(P_kW, 0)), 0) / 60.0, 3) AS centroid_hour_aest
    FROM se_interval
    WHERE month(ts_aest) IN (1, 6)
    GROUP BY 1, 2 ORDER BY 1, 2
""")
display(centroids.pivot(index="state", columns="season", values="centroid_hour_aest"))

## 6. Sites generating outside daylight hours

### Likely storage (5 sites, ~92% of the affected rows)

These inverters report **continuously**: ~105,000 rows a year, which is exactly
24 h × 12 intervals × 365, with overnight row counts matching midday row counts. Their
night output is a low, flat plateau that scales with season — AUS639 averages 0.46 kW
overnight in January and 0.10 kW in June, flat right across the night rather than
peaking anywhere.

### Stray timestamps (15 sites, ~8% of the affected rows)

These report daylight hours only (~53,000 rows a year) and carry a handful of night rows
— 3 to 323 across the whole year — at daytime power levels. This is the original
hypothesis, and for these sites it holds.

In [ ]:
anomaly = ing.night_generation_anomaly(con)
display(anomaly)

summary = anomaly.groupby("classification").agg(
    sites=("site_alias", "count"), night_rows=("n_night_rows", "sum"))
display(summary)

print("night_coverage_pct is the discriminator: overnight rows as a % of midday rows.")
print("~100% means the inverter logs all night (storage); ~0% means daylight-only.")
print("Sites near the 50% boundary are genuinely ambiguous - check them individually.")

anomaly.to_csv(C.ARTEFACT_DIR / "night_generation_sites.csv", index=False)
print(f"\nWritten to {C.ARTEFACT_DIR / 'night_generation_sites.csv'}")

## 7. Where the daylight-saving rows landed

A confirmation step for section 2, and the one piece of this notebook that needs its
frame stated carefully.

The transition is an **instant**, not a wall-clock hour, so this filters on `ts_utc`, not
`ts_aest`. The two DST changeovers occur at:

| Transition | UTC instant |
|---|---|
| April, clocks back (03:00 AEDT → 02:00 AEST) | 2025-04-05 16:00 |
| October, clocks forward (02:00 AEST → 03:00 AEDT) | 2025-10-04 16:00 |

The query takes a two-hour UTC window straddling each, and counts what landed there by
state. It is asking: *after conversion, did anything strange end up near the boundary?*

What to expect, and what the output shows: a handful of rows and essentially no
generation, because this is 02:00–03:00 local — inside the ~3% overnight coverage.
Queensland appears because it shares the UTC window, not because anything happened to it.

This overlaps deliberately with `dst_hazards()` in section 2. That one checks the **raw**
delivery *before* conversion; this checks the **store** *after*. Same question from both
ends — the check that matters most is section 2, since that is the one that can fail.

In [ ]:
display(ing.dst_audit(con))

## 8. Re-partitioning

The delivered files have a single row group each, so any predicate forces a full column
scan. The store is sorted by `(site_alias, ts_utc)` within month partitions, which is
what lets DuckDB skip row groups.

The single-site query is the one to watch — that access pattern drives every per-site
diagnostic and day plot from here on.

In [ ]:
import time

def timed(label, sql):
    start = time.perf_counter()
    n = con.execute(sql).fetchone()[0]
    return {"query": label, "seconds": round(time.perf_counter() - start, 3), "rows": n}

raw_files = C.duckdb_path_list(C.raw_files())
bench = pd.DataFrame([
    timed("RAW  : one site, whole year",
          f"SELECT count(*) FROM read_parquet({raw_files}) WHERE site_alias='AUS765'"),
    timed("STORE: one site, whole year",
          "SELECT count(*) FROM se_interval WHERE site_alias='AUS765'"),
    timed("STORE: one site, one month",
          "SELECT count(*) FROM se_interval WHERE site_alias='AUS765' AND dt_month='2025-06'"),
    timed("STORE: Volt-VAr band, whole year",
          "SELECT count(*) FROM se_interval "
          "WHERE V_max > 240 AND V_max < 253 AND hour(ts_aest) BETWEEN 11 AND 13"),
])
display(bench)
display(se_store.store_status(con)[["logical_name", "exists", "size_mb", "n_rows"]])